In [47]:
# Library imports
import torch
import math
import random
import json
import tqdm
import os
import spacy
import re

import numpy as np

from transformers import AutoTokenizer, AutoModelForSequenceClassification, AutoModelForCausalLM
from sentence_transformers import SentenceTransformer

from sklearn.metrics.pairwise import cosine_similarity

from dotenv import dotenv_values

from src.claim_extraction.config import DOTENV_PATH
from src.claim_extraction.extractor import extract_claims

In [48]:
# Load env settings

env = dotenv_values(DOTENV_PATH)

In [49]:
# Determine what device to use
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

if device.type == 'cuda':
    device_name = torch.cuda.get_device_name(device)

    print(f'CUDA on {device_name}')
else:
    print(f'CPU')

CUDA on NVIDIA GeForce RTX 3060 Laptop GPU


In [50]:
# Check if everything's cleaned
allocated = torch.cuda.memory_allocated() / (1024 * 1024)
reserved = torch.cuda.memory_reserved() / (1024 * 1024)

print(f'Allocated: {allocated:.1f} MB; Reserved: {reserved:.1f} MB')

Allocated: 1012.1 MB; Reserved: 1938.0 MB


In [90]:
# Configuration

DETECT_MODE = 'sentences'

NLI_MODEL = 'MoritzLaurer/DeBERTa-v3-large-mnli-fever-anli-ling-wanli' # https://huggingface.co/MoritzLaurer/DeBERTa-v3-large-mnli-fever-anli-ling-wanli
EMBED_MODEL = 'sentence-transformers/all-MiniLM-L6-v2' # https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2

NLTK_MODEL = 'tokenizers/punkt/english.pickle'

# Standard generation parameters used by all runs
CLAIM_MODEL_NAME = env.get('CLAIM_MODEL_1', 'Qwen/Qwen3.5-4B')
CLAIM_MODEL_TEMPERATURE = 0.1
CLAIM_MODEL_MAX_NEW_TOKENS = 4096
CLAIM_MODEL_BACKEND = 'local'
CLAIM_MODEL_VERBOSE = True

In [52]:
# Create models

nli_tokenizer = AutoTokenizer.from_pretrained(NLI_MODEL)
nli_model = AutoModelForSequenceClassification.from_pretrained(NLI_MODEL).to(device)

embed_model = SentenceTransformer(EMBED_MODEL).to(device)

nlp = spacy.load("en_core_web_sm")

Loading weights:   0%|          | 0/394 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: MoritzLaurer/DeBERTa-v3-large-mnli-fever-anli-ling-wanli
Key                             | Status     |  | 
--------------------------------+------------+--+-
deberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [53]:
# Load dataset

dataset = json.load(open('datasets/ContraDoc/ContraDoc.json', 'r'))

# Label examples
for example in dataset['pos'].values():
	example['label'] = 1.0 

for example in dataset['neg'].values():
	example['label'] = 0.0 

# Flatten list
all_examples = { **dataset['pos'], **dataset['neg'] }

print(f'Dataset loaded with {len(dataset["pos"])} positive and {len(dataset["neg"])} negative examples.')

Dataset loaded with 449 positive and 442 negative examples.


In [ ]:
# Model functions

def normalize_punctuation(text: str) -> str:
    # remove spaces before punctuation
    text = re.sub(r"\s+([,.;:!?])", r"\1", text)

    # ensure space after punctuation if followed by word
    text = re.sub(r"([,.;:!?])([A-Za-z])", r"\1 \2", text)

    # normalize quotes spacing
    text = re.sub(r"\s+([\"'])", r" \1", text)
    text = re.sub(r"([\"'])\s+", r"\1 ", text)

    # normalize dash spacing
    text = text.replace(" - ", "-")

    # collapse whitespace
    #text = re.sub(r"\s+", " ", text)

    return text.strip()

def remove_structural_lines(text):
    lines = text.split("\n")
    clean = []
    
    for line in lines:
        l = line.strip()

        if not l:
            continue

        # wikipedia headers
        if re.match(r"^\s*(=+\s*)+[^=]+?(\s*=)+\s*$", l):
            continue

        # markdown headers
        if re.match(r"^#+\s", l):
            continue

        # numbered lists
        if re.match(r"^\d+\s*[.)]\s*", l):
            continue

        # bullet lists
        if re.match(r"^[-*]\s+", l):
            continue

        # table rows
        if "|" in l and l.count("|") >= 2:
            continue

        clean.append(l)

    return " ".join(clean)

def normalize_numbers(text: str) -> str:
    # Weird commas
    text = text.replace("@,@", ",")

    # decimal numbers: "2 . 9" -> "2.9"
    text = re.sub(r"(\d)\s*\.\s*(\d)", r"\1.\2", text)

    # thousands separators: "1 , 000" -> "1,000"
    text = re.sub(r"(\d)\s*,\s*(\d)", r"\1,\2", text)

    # currency spacing "$ 2.9" -> "$2.9"
    text = re.sub(r"([$€£])\s+(\d)", r"\1\2", text)

    return text

def normalize_brackets(text: str) -> str:
    # "( text )" -> "(text)"
    text = re.sub(r"\(\s+", "(", text)
    text = re.sub(r"\s+\)", ")", text)

    text = re.sub(r"\[\s+", "[", text)
    text = re.sub(r"\s+\]", "]", text)

    text = re.sub(r"\{\s+", "{", text)
    text = re.sub(r"\s+\}", "}", text)

    return text

def has_verb(text):
    doc = nlp(text)
    return any(token.pos_ == "VERB" for token in doc)

def split_sentences(text):
    text = normalize_punctuation(text)
    text = remove_structural_lines(text)
    text = normalize_numbers(text)
    text = normalize_brackets(text)

    doc = nlp(text)
    sentences = [s.text.strip() for s in doc.sents]

    def sentence_filter(sentence):
        # min word count
        if len(sentence.split()) < 4:
            return False
        
        # contain actual words
        if re.match(r"^[\d\s.,-]+$", sentence):
            return False
        
        # requires verb
        if not has_verb(sentence):
            return False
        
        return True

    return list(filter(sentence_filter, sentences))

def argtopk(a, k):
    idx = np.argpartition(a, -k)[-k:]
    return idx[np.argsort(a[idx])[::-1]]

def get_sentence_clusters(sentences, embeddings, top_k):
    # Compute pairwise cosine similarity
    similarity_matrix = cosine_similarity(embeddings)
    
    for i in range(len(sentences)):
        similarity_matrix[i][i] = -1.0  # Exclude self-similarity by setting diagonal to -1

    cluster_indices = [ ]
    for i in range(len(sentences)):
        cluster_indices.append(argtopk(similarity_matrix[i], top_k))

    clusters = [ ]
    for i, neighbours in enumerate(cluster_indices):
        cluster = [ sentences[i], *[ sentences[j] for j in neighbours ]]
        clusters.append(cluster)

    return clusters

def predict(premise, hypothesis):
    input = nli_tokenizer(premise, hypothesis, truncation=True, return_tensors="pt").to(device)

    output = nli_model(input["input_ids"])
    prediction = torch.softmax(output["logits"][0], -1).tolist()
    label_names = ["entailment", "neutral", "contradiction"]

    return {name: float(pred) for pred, name in zip(prediction, label_names)}
    
def detect(id, document, mode, verbose):
    if mode == 'claims':
        file_name = f'./extracted_claims/{id}.txt'
        if os.path.isfile(file_name):
            with open(file_name) as file:
                claims = file.readlines()
        else:
            claims = extract_claims(
                text=document,
                model_name=CLAIM_MODEL_NAME,
                backend=CLAIM_MODEL_BACKEND,
                temperature=CLAIM_MODEL_TEMPERATURE,
                max_new_tokens=CLAIM_MODEL_MAX_NEW_TOKENS,
                verbose=CLAIM_MODEL_VERBOSE,
            )

    elif mode == 'sentences':
        claims = split_sentences(document)
    else:
        raise "Invalid mode"

    if verbose:
        print()
        print(f'Extracted {len(claims)} claims:')
        print('\n'.join(claims))

    if len(claims) <= 1:
        return None, None, None
    
    embeddings = embed_model.encode(claims)

    single_claims = [ [ claim ] for claim in claims ]
    clustered_claims = get_sentence_clusters(claims, embeddings, 2)
    clusters = [ *single_claims, *clustered_claims ]

    p_contra_max = 0.0
    evidence = []
    evidence_max = None

    for cluster in clusters:
        if len(cluster) > 1:
            p_contra = 0

            for i in range(len(cluster)):
                premise_claims = [ claim for j, claim in enumerate(cluster) if i != j ]
                premise = ' '.join(premise_claims)
                hypothesis = cluster[i]
            
                prediction = predict(premise, hypothesis)
                p_contra += prediction['contradiction']

                if verbose:
                    print(f'Comparing:')
                    
                    print(f'  {premise}')
                    print(f'  {hypothesis}')

                    print()

                    print(f'  Prediction: {prediction["contradiction"]:.4f}')
                    print()
            
            p_contra = p_contra / len(cluster)
        else:
            prediction = predict(cluster[0], "")
            p_contra = prediction['contradiction']

            if verbose:
                print(f'Testing:')
                print(f'  {cluster[0]}')
                print(f'  Prediction: {p_contra:.4f}')
                print()

        if p_contra > p_contra_max:
            evidence_max = (cluster, p_contra)
            p_contra_max = p_contra

        if p_contra > 0.70:
            evidence.append((cluster, p_contra))

    return p_contra_max, evidence_max, evidence

In [ ]:
# NLI test

contra = "Holmes grows angry when Watson touches items explaining that he doesn't mind his things being touched"

premise = contra
hypothesis = ""

print(predict(premise, hypothesis))

{'entailment': 0.533203125, 'neutral': 0.359619140625, 'contradiction': 0.107177734375}


In [94]:
# Single evaluation test

pos = True					# Test example from positive or negative set
example_id = None	# Specific example ID to test, or None for random

# 3489738232_6
# story_train_3585
# wiki_train_29443

if example_id is None:
	example_ids = list((dataset['pos' if pos else 'neg']).keys())
	example_id = example_ids[random.randint(0, len(example_ids) - 1)]

example = all_examples[example_id]

print(f'Detecting contradictions in example {example["unique id"]}')
print()

print(example['text'])
print()

if 'evidence' in example:
	print(f'Evidence: {example["evidence"]}')
	print(f'Ref sentences: {example.get("ref sentences", [])}')

print()

p_contra, evidence_max, all_evidence = detect(example['unique id'], example['text'], DETECT_MODE, True)
p_contra, evidence_max, all_evidence

Detecting contradictions in example 3488771849_5

Note: This synopsis is consistent with the novel in its later forms (1946 and subsequent editions) but differs in detail from the original 1928 text as transcribed at Project Gutenberg. There were significant changes between the 1928 magazine publication and the 1946 hardcover, and between the early hardcovers and the late 1950s and later paperback editions. The Skylark of Space is the first book of the Skylark series and pits the idealistic protagonist, Dick Seaton, against the mercantile antagonist Marc "Blackie" DuQuesne. At the beginning of the story, Seaton accidentally discovers a workable space drive in combining pure copper with a newly discovered [fictional] element "X" (suggested to be a stable transactinide element in the platinum group) in solution. Having failed to re-create the effect, Seaton realizes that the missing component is a field generated by DuQuesne's particle accelerator, and thereafter sets up a business with 

(0.8076171875,
 (['Having obtained the hostages, Seaton extracts a promise from DuQuesne to "act as one of the party until they get back to Earth", in which relationship they leave orbit and travel further in search of additional fuel.'],
  0.8076171875),
 [(['Having obtained the hostages, Seaton extracts a promise from DuQuesne to "act as one of the party until they get back to Earth", in which relationship they leave orbit and travel further in search of additional fuel.'],
   0.8076171875),
  (['On an Earthlike exoplanet, they fail to find any "X" and leave that planet in search of copper.'],
   0.7490234375),
  (["In the resulting fight, DuQuesne's ship is accidentally set to full acceleration on an uncontrolled trajectory, until the copper 'power bar' is exhausted at a vast distance from Earth's solar system.",
    'DuQuesne conspires to sabotage Seaton\'s spaceship and build his own from Seaton\'s plans, which he uses to kidnap Seaton\'s fiancée, Dorothy Vaneman, to exchange for 

In [101]:
# Evaluation loop

experiment_id = 'cluster_sentence_nli_r1'
output_file = f'./data/results.{experiment_id}.json'
resume = True

# Allow resuming from previous results
if resume:
	results = json.load(open(output_file, "r"))
	
	# Runs store lists instead of dicts, convert them
	if type(results) is list:
		results = { x['unique_id']: x for x in results }
else:
	results = { }

def write_output():
	# Write output file
	with open(output_file, "w") as f:
		json.dump(list(results.values()), f, indent=2)


# Create flat list of examples, shuffle to improve live stats accuracy
test_set = list(all_examples.items())
random.shuffle(test_set)

# Enumerate to get indices, convert to list to get progress indication
test_set = list(enumerate(test_set))

for i, (unique_id, example) in tqdm.tqdm(test_set):
	unique_id = example["unique id"]
	text = example["text"]
	y_true = example["label"]        # 0 or 1
	doc_type = example["doc_type"]   # story/wiki/news

	# Skip tests already completed previous run
	if unique_id in results:
		continue

	p_pred, evidence_max, all_evidence = detect(unique_id, text, DETECT_MODE, False)

	# Some examples fail due to claim extraction returning 0 or 1 claims
	if p_pred is None:
		print(f'Warning! Detection failed for {unique_id}')
		continue

	results[unique_id] = {
		"unique_id": unique_id,
		"p_pred": p_pred,
		"y_true": y_true,
		"doc_type": doc_type
	}

	# Ocassionally write output, to get live stats and in case of crash
	if i % 1 == 0:
		write_output()

write_output()

  0%|          | 0/891 [00:00<?, ?it/s]

['Hal Warner, a rich young fellow determined to find the truth for himself about conditions in the mines, runs away from home and adopts the alias "Joe Smith."', 'After being turned away by one coal mine for fear of Hal being a union organizer, he gets a job in another coal mine operated by the General Fuel Company, or GFC.', 'In the mines he befriends many of the workers, and realizes their misery and exploitation at the hands of the bosses.', "He befriends Mary Burke, who is a passionate fighter for the workers' rights.", 'Her father is a mine worker who spends his days drinking and leaving her to take care of her siblings.', "She and Hal grow close, which tears at Hal's loyalty to his fiancĂŠe back home.", "After dedicating himself to the workers' cause, he tells them that he will appeal to the bosses to become a check weigh man who measures the amount of coal, but the GFC, wanting to cheat the workers out of their pay, appoints a company check weigh man.", 'Hal is eventually put in

  0%|          | 4/891 [00:03<13:41,  1.08it/s]

['= I Miss You (Miley Cyrus song) = " I Miss You " is a song by American recording artist, Miley Cyrus.', 'It was co-written by Cyrus (credited under her birth name Destiny Hope Cyrus), Brian Green, Wendy Foy Green, and produced by Brian Green. "', 'I Miss You " is a homage to Cyrus \' late grandfather, Ron Cyrus, who died in February 28,2006.', 'He was diagnosed with mesothelioma, and, seeing her grandfather nearing death, Cyrus wanted to write him a song prior to his decease.', 'It was released to Radio Disney mid-year 2007 as promotion for the dual disc Hannah Montana 2: Meet Miley Cyrus.', 'The song received generally positive reviews from music critics; some commented on how it deviated from her usual material at the time and how effective the message was. "', 'I Miss You " appeared on two United States charts: it peaked at number nine on Bubbling Under Hot 100 Singles, an extension of the Billboard Hot 100 chart, and at number ninety-two on the defunct Pop 100.', 'Cyrus performed

 21%|██        | 187/891 [00:16<00:57, 12.28it/s]

...

 93%|█████████▎| 833/891 [01:53<00:13,  4.15it/s]

["The play opens with the recruiter, Captain Plume's Sergeant Kite, recruiting in the town of Shrewsbury.", "Plume arrives, in love with Sylvia, closely followed by Worthy, a local gentleman who is in love with Sylvia's cousin Melinda.", 'Worthy asked Melinda to become his mistress a year previously, as he believed her to be of inadequate fortune to marry.', 'But he changes his mind after she comes into an inheritance of ÂŁ20,000.', "Melinda accepts an invitation from Captain Brazen, another recruiter, to annoy Worthy, as she was offended by Worthy's previous offer.", 'However, her maid Lucy meets Brazen, pretending to be Melinda, hoping to marry him herself.', 'Melinda and Sylvia argue after Melinda says that the money she has inherited makes her more desirable.', "Silvia, who is more down to earth, is infuriated by Melinda's newly haughty behaviour.", "Sylvia leaves her father's house to mourn her brother Owen's death.", "She tells her father Balance that she is going to the Welsh co

 97%|█████████▋| 863/891 [01:58<00:06,  4.47it/s]

['Villette begins with its famously passive protagonist, Lucy Snowe, age 14, staying at the home of her godmother Mrs. Bretton in "the clean and ancient town of Bretton", in England.', "Also in residence are Mrs. Bretton's son, John Graham Bretton (whom the family calls Graham), and a young visitor, Paulina Home (who is called Polly).", 'Polly is a peculiar little girl who soon develops a deep devotion to Graham, who showers her with attention.', "But Polly's visit is cut short when her father arrives to take her away.", "For reasons that are not stated, Lucy leaves Mrs. Bretton's home a few weeks after the Polly's departure.", 'Some years pass, during which an unspecified family tragedy leaves Lucy without family, home, or means.', 'After some initial hesitation, she is hired as a caregiver by Miss Marchmont, a rheumatic crippled woman.', 'Lucy is soon accustomed to her work and has begun to feel content with her quiet lifestyle.', 'During an evening of dramatic weather changes, Miss 

100%|██████████| 891/891 [02:07<00:00,  7.00it/s]
